In [1]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

context = "The capital of France is Paris. It is famous for the Eiffel Tower."
question = "What is the capital of France?"

basic_enc = tokenizer(question, context, return_tensors="pt")
print("Keys:", list(basic_enc.keys()))
print("input_ids:", basic_enc["input_ids"][0].tolist())
print("tokens:", tokenizer.convert_ids_to_tokens(basic_enc["input_ids"][0]))
print("attention_mask:", basic_enc["attention_mask"][0].tolist())
if "token_type_ids" in basic_enc:
  print("token_type_ids:", basic_enc["token_type_ids"][0].tolist())
else:
  print("Model does not use token_type_ids")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 2009, 2003, 3297, 2005, 1996, 1041, 13355, 2884, 3578, 1012, 102]
tokens: ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', 'it', 'is', 'famous', 'for', 'the', 'e', '##iff', '##el', 'tower', '.', '[SEP]']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [2]:
answer_text = "Paris"
char_start = context.index(answer_text)
char_end = char_start + len(answer_text)
print(f"Answer: {answer_text!r} | char span: [{char_start}, {char_end})")
print("Slice by char:", context[char_start:char_end])

words = context.split()
word_idx = next((i for i, w in enumerate(words) if answer_text in w), None)
print("Word index in context.split():", word_idx)

ctx_only = tokenizer(
    context, return_offsets_mapping=True, add_special_tokens=False
)
for t_idx, (tok, (s, e)) in enumerate(
    zip(ctx_only.tokens(), ctx_only["offset_mapping"])
):
  if (
      s <= char_start < e
      or s < char_end <= e
      or (char_start <= s and e <= char_end)
  ):
    print(f"Token idx={t_idx}, token={tok!r}, offset=({s}, {e})")

Answer: 'Paris' | char span: [25, 30)
Slice by char: Paris
Word index in context.split(): 5
Token idx=5, token='paris', offset=(25, 30)


In [3]:
long_context = " ".join([
    "Paris is the capital and most populous city of France.",
    (
        "It has been one of Europe’s major centres of finance, diplomacy,"
        " commerce, fashion, gastronomy, and science."
    ),
    (
        "The City of Light is also well known for its museums and architectural"
        " landmarks such as the Louvre and Eiffel Tower."
    ),
    (
        "In extractive question answering, long passages exceeding max_length"
        " are split into overlapping windows using stride."
    ),
])
q_long = "What is Paris well known for?"
ans_long = "museums and architectural landmarks"
ans_s = long_context.index(ans_long)
ans_e = ans_s + len(ans_long)

win_enc = tokenizer(
    q_long,
    long_context,
    truncation="only_second",
    max_length=45,
    stride=15,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length",
)

print("Total windows generated:", len(win_enc["input_ids"]))
print("overflow_to_sample_mapping:", win_enc["overflow_to_sample_mapping"])
for w_i in range(len(win_enc["input_ids"])):
  toks = tokenizer.convert_ids_to_tokens(win_enc["input_ids"][w_i])
  print(
      f"Window {w_i} (sample {win_enc['overflow_to_sample_mapping'][w_i]}):",
      toks[:10],
      "...",
  )

Total windows generated: 4
overflow_to_sample_mapping: [0, 0, 0, 0]
Window 0 (sample 0): ['[CLS]', 'what', 'is', 'paris', 'well', 'known', 'for', '?', '[SEP]', 'paris'] ...
Window 1 (sample 0): ['[CLS]', 'what', 'is', 'paris', 'well', 'known', 'for', '?', '[SEP]', 'centres'] ...
Window 2 (sample 0): ['[CLS]', 'what', 'is', 'paris', 'well', 'known', 'for', '?', '[SEP]', 'light'] ...
Window 3 (sample 0): ['[CLS]', 'what', 'is', 'paris', 'well', 'known', 'for', '?', '[SEP]', '.'] ...


In [4]:
def map_char_span_to_token_span(offsets, sequence_ids, char_start, char_end):
  ctx_indices = [i for i, sid in enumerate(sequence_ids) if sid == 1]
  if not ctx_indices:
    return None, "no_context_in_window"

  first_ctx = ctx_indices[0]
  last_ctx = ctx_indices[-1]
  win_start_char = offsets[first_ctx][0]
  win_end_char = offsets[last_ctx][1]

  if char_end <= win_start_char or char_start >= win_end_char:
    return None, "outside_window_or_truncated"

  start_tok = None
  end_tok = None
  for idx in ctx_indices:
    s, e = offsets[idx]
    if s == e:
      continue
    if s <= char_start < e:
      start_tok = idx
    if s < char_end <= e:
      end_tok = idx

  status = "exact_aligned"
  if start_tok is not None and end_tok is None:
    end_tok = last_ctx
    status = "truncated_end_in_window"
  elif start_tok is None and end_tok is not None:
    start_tok = first_ctx
    status = "truncated_start_in_window"
  elif start_tok is None and end_tok is None:
    return None, "span_not_matched"

  return (start_tok, end_tok), status


for w_i in range(len(win_enc["input_ids"])):
  seq_ids = win_enc.sequence_ids(w_i)
  offsets = win_enc["offset_mapping"][w_i]
  span, status = map_char_span_to_token_span(offsets, seq_ids, ans_s, ans_e)
  print(f"Window {w_i}: span={span}, status={status}")

Window 0: span=None, status=outside_window_or_truncated
Window 1: span=(36, 39), status=exact_aligned
Window 2: span=(16, 19), status=exact_aligned
Window 3: span=None, status=outside_window_or_truncated
